# 4. Modelagem com XGBoost

## Objetivo

Treinar e avaliar um modelo de predição de óbito hospitalar por COVID-19 usando **XGBoost** (eXtreme Gradient Boosting).

## O que é XGBoost?

**XGBoost** é um algoritmo de gradient boosting otimizado que:
- É mais rápido e eficiente que implementações anteriores
- Tem regularização integrada para evitar overfitting
- Oferece melhor desempenho em competições de machine learning
- Fornece importância das features

### Diferença entre LightGBM e XGBoost
Ambos são gradient boosting, mas:
- **LightGBM**: Mais rápido, usa menos memória, melhor para datasets grandes
- **XGBoost**: Mais robusto, melhor regularização, mais usado em produção

---

## Seção 1: Importações e Carregamento de Dados

In [ ]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json

# XGBoost
import xgboost as xgb

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, auc
)

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✓ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar dados processados
processed_dir = Path("../data/processed")

print("Carregando dados processados...")
X_train = pd.read_csv(processed_dir / "X_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv").squeeze()
y_test = pd.read_csv(processed_dir / "y_test.csv").squeeze()

print(f"✓ Dados carregados com sucesso!")
print(f"\nConjunto de treino: {X_train.shape}")
print(f"Conjunto de teste: {X_test.shape}")

## Seção 2: Treinamento do Modelo XGBoost

In [ ]:
print("="*80)
print("TREINAMENTO DO MODELO XGBOOST")
print("="*80)

# Definir hiperparâmetros
params = {
    'objective': 'binary:logistic',  # Classificação binária
    'eval_metric': 'auc',  # Métrica de avaliação
    'max_depth': 6,  # Profundidade máxima
    'learning_rate': 0.05,  # Taxa de aprendizado
    'subsample': 0.8,  # Usar 80% das amostras
    'colsample_bytree': 0.8,  # Usar 80% das features
    'min_child_weight': 1,  # Mínimo peso em uma folha
    'gamma': 0,  # Redução mínima de perda
    'reg_alpha': 0,  # Regularização L1
    'reg_lambda': 1,  # Regularização L2
    'scale_pos_weight': 1,  # Peso para classe positiva
    'random_state': 42,
    'verbosity': 0
}

print("\nHiperparâmetros:")
for key, value in params.items():
    print(f"  {key}: {value}")

# Criar DMatrix (formato otimizado do XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

print("\n✓ DMatrix criadas")

# Treinar modelo
print("\nTreinando modelo...")
evals = [(dtrain, 'train'), (dtest, 'eval')]
evals_result = {}

model_xgb = xgb.train(
    params,
    dtrain,
    num_boost_round=100,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=10,
    verbose_eval=20
)

print("\n✓ Modelo treinado com sucesso!")

## Seção 3: Avaliação do Modelo

In [ ]:
# Fazer previsões
print("="*80)
print("AVALIAÇÃO DO MODELO XGBOOST")
print("="*80)

# Previsões de probabilidade
y_pred_proba_train = model_xgb.predict(dtrain)
y_pred_proba_test = model_xgb.predict(dtest)

# Previsões binárias
y_pred_train = (y_pred_proba_train >= 0.5).astype(int)
y_pred_test = (y_pred_proba_test >= 0.5).astype(int)

print("\nPrevisões realizadas!")
print(f"Distribuição de previsões no teste:")
print(f"  Sobreviveu (0): {(y_pred_test == 0).sum()}")
print(f"  Óbito (1): {(y_pred_test == 1).sum()}")

In [ ]:
# Calcular métricas
metrics_train = {
    'Acurácia': accuracy_score(y_train, y_pred_train),
    'Precisão': precision_score(y_train, y_pred_train, zero_division=0),
    'Recall': recall_score(y_train, y_pred_train, zero_division=0),
    'F1-Score': f1_score(y_train, y_pred_train, zero_division=0),
    'AUC-ROC': roc_auc_score(y_train, y_pred_proba_train)
}

metrics_test = {
    'Acurácia': accuracy_score(y_test, y_pred_test),
    'Precisão': precision_score(y_test, y_pred_test, zero_division=0),
    'Recall': recall_score(y_test, y_pred_test, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_test, zero_division=0),
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba_test)
}

# Exibir métricas
print("\nMÉTRICAS DE DESEMPENHO - CONJUNTO DE TREINO:")
for metric, value in metrics_train.items():
    print(f"  {metric}: {value:.4f}")

print("\nMÉTRICAS DE DESEMPENHO - CONJUNTO DE TESTE:")
for metric, value in metrics_test.items():
    print(f"  {metric}: {value:.4f}")

# Comparação
print("\nCOMPARAÇÃO TREINO vs TESTE:")
for metric in metrics_train.keys():
    diff = metrics_train[metric] - metrics_test[metric]
    print(f"  {metric}: Treino={metrics_train[metric]:.4f}, Teste={metrics_test[metric]:.4f}, Diferença={diff:.4f}")

In [ ]:
# Matriz de confusão
print("\nMATRIZ DE CONFUSÃO - TESTE:")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

# Visualizar matriz de confusão
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax,
            xticklabels=['Sobreviveu', 'Óbito'],
            yticklabels=['Sobreviveu', 'Óbito'])
ax.set_xlabel('Previsão')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão - XGBoost (Teste)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/13_xgboost_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='green', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classificador Aleatório')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.set_title('Curva ROC - XGBoost (Teste)', fontsize=12, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/14_xgboost_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")
print("\n✓ Gráfico salvo")

## Seção 4: Importância das Features

In [ ]:
# Importância das features
print("="*80)
print("IMPORTÂNCIA DAS FEATURES - XGBOOST")
print("="*80)

# Obter importância
importance_dict = model_xgb.get_score(importance_type='weight')
feature_importance = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    'Importância': list(importance_dict.values())
}).sort_values('Importância', ascending=False)

print("\nTop 15 Features mais importantes:")
print(feature_importance.head(15).to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(12, 8))
top_features = feature_importance.head(15)
ax.barh(range(len(top_features)), top_features['Importância'], color='seagreen')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.set_xlabel('Importância (Weight)')
ax.set_title('Top 15 Features mais Importantes - XGBoost', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/15_xgboost_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

## Seção 5: Salvando o Modelo

In [ ]:
# Salvar modelo
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

# Salvar modelo XGBoost
model_xgb.save_model(str(results_dir / "xgboost_model.json"))

# Salvar métricas
metrics_summary = {
    'algorithm': 'XGBoost',
    'train_metrics': metrics_train,
    'test_metrics': metrics_test,
    'feature_importance': feature_importance.to_dict('list')
}

with open(results_dir / "xgboost_metrics.json", 'w') as f:
    json.dump(metrics_summary, f, indent=4)

# Salvar feature importance
feature_importance.to_csv(results_dir / "xgboost_feature_importance.csv", index=False)

print("="*80)
print("MODELO E RESULTADOS SALVOS")
print("="*80)

print(f"\n✓ xgboost_model.json: Modelo treinado")
print(f"✓ xgboost_metrics.json: Métricas de desempenho")
print(f"✓ xgboost_feature_importance.csv: Importância das features")
print(f"\nLocalização: {results_dir.absolute()}")

## Seção 6: Resumo e Conclusões

In [ ]:
print("\n" + "="*80)
print("RESUMO - MODELAGEM COM XGBOOST")
print("="*80)

print(f"""
1. MODELO TREINADO:
   ✓ Algoritmo: XGBoost (eXtreme Gradient Boosting)
   ✓ Tipo: Classificação binária
   ✓ Variáveis: {X_train.shape[1]}
   ✓ Amostras de treino: {X_train.shape[0]:,}
   ✓ Amostras de teste: {X_test.shape[0]:,}

2. DESEMPENHO NO TESTE:
   ✓ Acurácia: {metrics_test['Acurácia']:.4f} ({metrics_test['Acurácia']*100:.2f}%)
   ✓ Precisão: {metrics_test['Precisão']:.4f}
   ✓ Recall: {metrics_test['Recall']:.4f}
   ✓ F1-Score: {metrics_test['F1-Score']:.4f}
   ✓ AUC-ROC: {metrics_test['AUC-ROC']:.4f}

3. FEATURES MAIS IMPORTANTES:
   Top 3: {', '.join(feature_importance.head(3)['Feature'].tolist())}

4. PRÓXIMOS PASSOS:
   → Treinar CatBoost
   → Comparar desempenho dos três algoritmos
   → Análise SHAP para explicabilidade
   → Análise de justiça algorítmica
""")

print("="*80)
print("Modelagem com XGBoost concluída! Prossiga para o notebook 05_modeling_catboost.ipynb")
print("="*80)